In [ ]:

# ============================================================
# Environment + Imports (Prediction/Ensemble)
# ============================================================
import os, sys, time, gc, warnings, json, zipfile, hashlib, traceback
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_ckpt_fn

import scipy.ndimage as ndi
from scipy.ndimage import (distance_transform_edt, generate_binary_structure,
                           label as cc_label, binary_erosion, binary_dilation,
                           binary_opening, gaussian_filter)

try:
    import tifffile
    def read_tif(path):
        return tifffile.imread(path)
    def write_tif(path, data):
        tifffile.imwrite(path, data)
except ImportError:
    from PIL import Image
    def read_tif(path):
        img = Image.open(path)
        frames = []
        try:
            while True:
                frames.append(np.array(img))
                img.seek(img.tell() + 1)
        except EOFError:
            pass
        return np.stack(frames, axis=0)
    def write_tif(path, data):
        from PIL import Image as PILImage
        imgs = [PILImage.fromarray(data[z]) for z in range(data.shape[0])]
        imgs[0].save(path, save_all=True, append_images=imgs[1:])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[ENV] PyTorch {torch.__version__}, Device: {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"[ENV] GPU: {props.name}, VRAM: {props.total_memory / 1e9:.1f} GB")

T_START = time.time()
def elapsed_h():
    return (time.time() - T_START) / 3600.0
def budget_ok(max_h=8.5):
    return elapsed_h() < max_h


In [ ]:

# ============================================================
# Configuration + Explicit Model Paths
# ============================================================
NUM_CLASSES = 2
IGNORE_LABEL = 255
IN_CHANNELS = 1
FEATURES = [32, 64, 128, 256, 320, 320]
BLOCKS = [1, 3, 4, 6, 6, 6]
STRIDES = [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]]
DEC_CONVS = [1, 1, 1, 1, 1]

# Competition paths
ROOT_CANDS = [
    "/kaggle/input/competitions/vesuvius-challenge-surface-detection",
    "/kaggle/input/vesuvius-challenge-surface-detection",
]
ROOT_DIR = next((p for p in ROOT_CANDS if os.path.exists(p)), None)
if ROOT_DIR is None:
    ROOT_DIR = "/kaggle/input/competitions/vesuvius-challenge-surface-detection"
TEST_DIR = os.path.join(ROOT_DIR, "test_images")

# Explicit model paths. Edit dataset_slugs to match your Kaggle dataset mounts.
MODEL_DATASET_MAP = {
    "model_a": {
        "dataset_slugs": ["vesuvius-model-a", "vesuvius-model-a-ckpt"],
        # v2.5: Try best_combo first (competition-metric-optimized), fallback to best
        "ckpt_candidates": ["model_a_best_combo.pt", "model_a_best.pt"],
        "meta": "model_a_meta.json",
    },
    "model_b": {
        "dataset_slugs": ["vesuvius-model-b", "vesuvius-model-b-ckpt"],
        "ckpt_candidates": ["model_b_best_combo.pt", "model_b_best.pt"],
        "meta": "model_b_meta.json",
    },
    "model_c": {
        "dataset_slugs": ["vesuvius-model-c", "vesuvius-model-c-ckpt"],
        "ckpt_candidates": ["model_c_best_combo.pt", "model_c_best.pt"],
        "meta": "model_c_meta.json",
    },
}

MODEL_PATHS = {}
META_PATHS = {}

for mname, spec in MODEL_DATASET_MAP.items():
    found = False
    for slug in spec["dataset_slugs"]:
        for sub in ["checkpoints", ""]:
            base = os.path.join("/kaggle/input", slug, sub) if sub else os.path.join("/kaggle/input", slug)
            # v2.5: Try checkpoint candidates in priority order
            for ckpt_name in spec.get("ckpt_candidates", [spec.get("ckpt", f"{mname}_best.pt")]):
                ckpt = os.path.join(base, ckpt_name)
                if os.path.exists(ckpt):
                    MODEL_PATHS[mname] = ckpt
                    meta = os.path.join(base, spec["meta"])
                    if os.path.exists(meta):
                        META_PATHS[mname] = meta
                    found = True
                    break
            if found:
                break
        if found:
            break
    if not found:
        print(f"[WARN] {mname} not found in any of {spec['dataset_slugs']}")

assert len(MODEL_PATHS) > 0, (
    f"FATAL: No model checkpoints found! Expected datasets: "
    f"{[s for spec in MODEL_DATASET_MAP.values() for s in spec['dataset_slugs']]}\n"
    f"Mount your training output datasets and update MODEL_DATASET_MAP slugs."
)

print(f"[CFG] Loaded {len(MODEL_PATHS)} model path(s):")
for k, v in MODEL_PATHS.items():
    sz = os.path.getsize(v) / 1e6
    print(f"  {k}: {v} ({sz:.1f} MB)")

# Inference config
DEFAULT_ROI = (192, 192, 192)
INF_OVERLAP = 0.5
USE_TTA = True
MAX_PRED_HOURS = 8.5

# v2.3: Ensemble fusion mode
# "logit_weighted" = logit-space weighted average → single postproc (primary)
# "vote" = per-model binarize+cleanup → majority vote (fallback)
ENS_MODE = "logit_weighted"

# v2.3: Role-based ensemble weights (structure > boundary > connectivity)
# These are base multipliers; actual weights = role_weight * val_dice
ROLE_WEIGHTS = {"structure": 1.5, "boundary": 1.2, "connectivity": 1.0}

# Confidence sharpening: temperature < 1.0 sharpens logits before fusion
CONF_TEMPERATURE = 0.85

# Post-processing (researcher's specific values for competition dominance)
PP_DUST_MIN_3D = 192           # Remove 3D dust < 192 voxels (was 64)
PP_HOLE_MAX_2D = 48            # Fill 2D holes <= 48 pixels (was 64)
PP_OPEN_R_XY = 1               # XY opening radius (per-slice)
PP_OPEN_R_3D = 1               # 3D opening radius=1, 1 iteration
PP_SMOOTH_SIGMA = 0.3
PP_BRIDGE_KILL = True

# Bridge-killer parameters (researcher's values)
PP_BK_NECK_R = 1               # Erosion radius to break thin necks
PP_BK_MIN_LOBE = 10000         # Components smaller than this after erosion get killed
PP_BK_MIN_NECK_LEN = 2         # Min neck length (erosion iterations)

# Fallback guards: auto-adjust thresholds for degenerate predictions
PP_EMPTY_TL_DROP = 0.08        # Lower tl by this if mask is empty
PP_EMPTY_TH_DROP = 0.10        # Lower th by this if mask is empty
PP_OVERFULL_FRAC = 0.45        # If FG fraction > this, mask is overfull
PP_OVERFULL_TH_RAISE = 0.05    # Raise th by this if overfull

# Default thresholds (researcher's recommended starting point)
DEFAULT_TL = 0.34
DEFAULT_TH = 0.62

# Auto-drop: exclude models with val_dice below this
AUTO_DROP_THRESH = 0.25

# Test volumes
test_paths = sorted([os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR) if f.endswith(".tif")]) \
             if os.path.isdir(TEST_DIR) else []
test_ids = [os.path.splitext(os.path.basename(p))[0] for p in test_paths]
print(f"[CFG] {len(test_ids)} test volumes, fusion={ENS_MODE}, temp={CONF_TEMPERATURE}")
print(f"[CFG] PP: dust={PP_DUST_MIN_3D}, hole={PP_HOLE_MAX_2D}, open_xy={PP_OPEN_R_XY}, "
      f"open_3d={PP_OPEN_R_3D}, bridge_kill={PP_BRIDGE_KILL}")


In [ ]:

# ============================================================
# Architecture (identical to training)
# ============================================================

class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, bias=True):
        super().__init__()
        if isinstance(kernel_size, int): kernel_size = [kernel_size] * 3
        if isinstance(stride, int): stride = [stride] * 3
        padding = [k // 2 for k in kernel_size]
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=bias)
        self.norm = nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(inplace=True)
    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

class ResBlock3D(nn.Module):
    def __init__(self, ch, kernel_size=3, bias=True):
        super().__init__()
        p = kernel_size // 2
        self.conv1 = nn.Conv3d(ch, ch, kernel_size, padding=p, bias=bias)
        self.norm1 = nn.InstanceNorm3d(ch, eps=1e-5, affine=True)
        self.conv2 = nn.Conv3d(ch, ch, kernel_size, padding=p, bias=bias)
        self.norm2 = nn.InstanceNorm3d(ch, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(inplace=True)
    def forward(self, x):
        r = x
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return self.act(x + r)

class ResidualEncoderUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2,
                 features=(32,64,128,256,320,320), blocks=(1,3,4,6,6,6),
                 strides=((1,1,1),(2,2,2),(2,2,2),(2,2,2),(2,2,2),(2,2,2)),
                 dec_convs=(1,1,1,1,1), deep_supervision=False, use_grad_ckpt=False):
        super().__init__()
        self.deep_supervision = deep_supervision
        self._ckpt = use_grad_ckpt
        n = len(features)
        self.enc = nn.ModuleList()
        for s in range(n):
            in_c = in_channels if s == 0 else features[s-1]
            out_c = features[s]
            st = list(strides[s]) if isinstance(strides[s], (list,tuple)) else [strides[s]]*3
            layers = [ConvBlock3D(in_c, out_c, 3, stride=st)]
            for _ in range(blocks[s]-1):
                layers.append(ResBlock3D(out_c, 3))
            self.enc.append(nn.Sequential(*layers))
        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        self.seg = nn.ModuleList()
        for i in range(n-1):
            s = n-1-i
            enc_ch, skip_ch, out_ch = features[s], features[s-1], features[s-1]
            st = list(strides[s]) if isinstance(strides[s], (list,tuple)) else [strides[s]]*3
            self.up.append(nn.ConvTranspose3d(enc_ch, enc_ch, kernel_size=st, stride=st, bias=True))
            nc = dec_convs[i] if i < len(dec_convs) else 1
            dl = []
            for c in range(nc):
                dl.append(ConvBlock3D((enc_ch+skip_ch) if c==0 else out_ch, out_ch, 3))
            self.dec.append(nn.Sequential(*dl))
            self.seg.append(nn.Conv3d(out_ch, num_classes, 1))

    def forward(self, x):
        skips = []
        for i, enc in enumerate(self.enc):
            x = enc(x)
            skips.append(x)
        outputs = []
        x = skips[-1]
        for i, (u, d, s) in enumerate(zip(self.up, self.dec, self.seg)):
            skip = skips[-(i+2)]
            x = u(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = d(x)
            outputs.append(s(x))
        outputs = outputs[::-1]
        if self.deep_supervision and self.training:
            return outputs
        return outputs[0]


In [ ]:

# ============================================================
# Load Models (strict loading, integrity checks)
# ============================================================
models = {}
metas = {}

for mname, ckpt_path in MODEL_PATHS.items():
    print(f"\n[LOAD] {mname} from {ckpt_path}...")
    m = ResidualEncoderUNet(
        in_channels=IN_CHANNELS, num_classes=NUM_CLASSES,
        features=FEATURES, blocks=BLOCKS, strides=STRIDES,
        dec_convs=DEC_CONVS, deep_supervision=False, use_grad_ckpt=False
    )

    sd = torch.load(ckpt_path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]

    missing, unexpected = m.load_state_dict(sd, strict=False)
    if unexpected:
        raise RuntimeError(
            f"[FATAL] {mname}: {len(unexpected)} UNEXPECTED keys in checkpoint!\n"
            f"  First 5: {unexpected[:5]}\n"
            f"  Check that you mounted the correct dataset."
        )
    if missing:
        non_seg_missing = [k for k in missing if ".seg." not in k]
        if non_seg_missing:
            raise RuntimeError(
                f"[FATAL] {mname}: {len(non_seg_missing)} critical missing keys!\n"
                f"  First 5: {non_seg_missing[:5]}"
            )
        print(f"  [WARN] {mname}: {len(missing)} missing keys (seg heads from DS)")

    m = m.to(DEVICE).eval()
    m = m.half() if DEVICE.type == "cuda" else m
    models[mname] = m

    # Load and validate meta
    meta_path = META_PATHS.get(mname)
    if meta_path and os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        assert meta.get("model") == mname, f"Meta model name mismatch: {meta.get('model')} vs {mname}"
        metas[mname] = meta
        print(f"  Meta: fold={meta.get('fold')}, val_dice={meta.get('best_val_dice', '?')}, "
              f"tl={meta.get('tl', '?')}, th={meta.get('th', '?')}, "
              f"temp={meta.get('temperature', '?')}, patch={meta.get('patch_size', '?')}")
    else:
        print(f"  [WARN] No meta found for {mname}, using defaults")
        metas[mname] = {"model": mname}

    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    _h = hashlib.sha1()
    for k, v in sorted(m.state_dict().items()):
        _h.update(k.encode())
        _h.update(str(tuple(v.shape)).encode())
    ckpt_hash = _h.hexdigest()[:12]
    print(f"  {mname}: {n_params:.1f}M params, hash={ckpt_hash}")

# v2.2: Auto-drop weak models
active_models = {}
for mname, m in models.items():
    # v2.4b: Use combined_proxy as primary trust metric (falls back to val_dice)
    meta = metas.get(mname, {})
    proxy = meta.get("combined_proxy", meta.get("best_val_dice", 0.5))
    if proxy < AUTO_DROP_THRESH and len(models) > 1:
        print(f"  [DROP] {mname}: proxy={proxy:.4f} < {AUTO_DROP_THRESH}, excluding from ensemble")
    else:
        active_models[mname] = m

if len(active_models) == 0:
    print("[WARN] All models below threshold, using all models anyway")
    active_models = dict(models)

models = active_models

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print(f"\n[LOAD] {len(models)} model(s) active for inference")


In [ ]:

# ============================================================
# Post-Processing: hysteresis + dust + holes + 3D/2D opening
#                  + bridge-killer + fallback guards
# Researcher-tuned parameters for competition dominance.
# ============================================================

def hysteresis_3d(prob, tl, th):
    """3D hysteresis thresholding: keep weak regions only if connected to strong."""
    struct26 = generate_binary_structure(3, 3)
    strong = prob >= th
    weak = prob >= tl
    cc, n = cc_label(weak, structure=struct26)
    if n == 0:
        return np.zeros_like(prob, dtype=np.uint8)
    strong_ids = np.unique(cc[strong])
    strong_ids = strong_ids[strong_ids != 0]
    return np.isin(cc, strong_ids).astype(np.uint8)


def remove_dust_3d(mask, min_size=PP_DUST_MIN_3D):
    """Remove small 3D connected components (26-connectivity)."""
    if min_size <= 0:
        return mask
    struct26 = generate_binary_structure(3, 3)
    cc, n = cc_label(mask, structure=struct26)
    if n == 0:
        return mask
    sizes = np.bincount(cc.ravel())
    keep = np.zeros(n + 1, dtype=bool)
    for i in range(1, n + 1):
        if sizes[i] >= min_size:
            keep[i] = True
    return (keep[cc]).astype(np.uint8)


def fill_holes_2d(mask, max_area=PP_HOLE_MAX_2D):
    """Fill small background holes slice-by-slice (XY plane)."""
    if max_area <= 0:
        return mask
    out = mask.copy()
    struct2d = generate_binary_structure(2, 1)
    for z in range(out.shape[0]):
        sl = out[z].astype(bool)
        bg = ~sl
        lbl, n = cc_label(bg, structure=struct2d)
        if n == 0:
            continue
        border = set()
        border.update(lbl[0, :].tolist())
        border.update(lbl[-1, :].tolist())
        border.update(lbl[:, 0].tolist())
        border.update(lbl[:, -1].tolist())
        for k in range(1, n + 1):
            if k in border:
                continue
            area = int((lbl == k).sum())
            if area <= max_area:
                sl[lbl == k] = True
        out[z] = sl.astype(np.uint8)
    return out


def xy_opening(mask, radius=PP_OPEN_R_XY):
    """Morphological opening slice-by-slice (removes thin XY protrusions)."""
    if radius <= 0:
        return mask
    struct2d = generate_binary_structure(2, 1)
    out = np.empty_like(mask)
    for z in range(mask.shape[0]):
        sl = mask[z].astype(bool)
        sl = binary_erosion(sl, structure=struct2d, iterations=radius)
        sl = binary_dilation(sl, structure=struct2d, iterations=radius)
        out[z] = sl.astype(np.uint8)
    return out


def opening_3d(mask, radius=PP_OPEN_R_3D):
    """3D morphological opening (erode -> dilate). Removes thin 3D bridges/spurs.
    Radius=1, 1 iteration as per researcher recommendation."""
    if radius <= 0:
        return mask
    struct6 = generate_binary_structure(3, 1)  # 6-connectivity for clean opening
    m = mask.astype(bool)
    m = binary_erosion(m, structure=struct6, iterations=radius)
    m = binary_dilation(m, structure=struct6, iterations=radius)
    return m.astype(np.uint8)


def bridge_killer(mask, neck_radius=PP_BK_NECK_R, min_lobe_size=PP_BK_MIN_LOBE,
                  min_neck_len=PP_BK_MIN_NECK_LEN):
    """
    Enhanced bridge killer with researcher-tuned parameters.
    1. Erode by neck_radius * min_neck_len to break thin necks
    2. Label components in eroded mask
    3. Drop components smaller than min_lobe_size (tiny fragments)
    4. Dilate each surviving component independently within original boundary
    Bridges stay broken because components dilate independently.
    Parameters: neck_radius=1, min_lobe_size=10000, min_neck_len=2
    """
    struct6 = generate_binary_structure(3, 1)    # 6-connectivity
    struct26 = generate_binary_structure(3, 3)   # 26-connectivity

    # Erode to break necks of width <= 2*neck_radius
    n_erosion_iters = max(1, neck_radius * min_neck_len)
    eroded = binary_erosion(mask.astype(bool), structure=struct6, iterations=n_erosion_iters)
    if not eroded.any():
        # Erosion removed everything -> try gentler (1 iteration)
        eroded = binary_erosion(mask.astype(bool), structure=struct6, iterations=1)
        if not eroded.any():
            return mask  # Keep original if even gentle erosion kills it

    cc_eroded, n_eroded = cc_label(eroded, structure=struct26)
    sizes = np.bincount(cc_eroded.ravel())

    result = np.zeros_like(mask, dtype=np.uint8)
    for i in range(1, n_eroded + 1):
        # Drop tiny lobes (fragments smaller than min_lobe_size)
        if sizes[i] < min_lobe_size:
            continue
        comp = (cc_eroded == i)
        # Dilate this component back within original mask boundary
        grown = binary_dilation(comp, structure=struct6, iterations=n_erosion_iters)
        grown = grown & mask.astype(bool)
        result[grown] = 1

    # Safety: if bridge-killer removed too much (>80%), keep original
    if result.sum() < mask.sum() * 0.2:
        print(f"    [BK] Bridge-killer too aggressive ({result.sum()} vs {mask.sum()}), keeping original")
        return mask

    return result


def postprocess(prob, tl, th, verbose=True):
    """Full post-processing pipeline with fallback guards.
    Order: smooth -> hysteresis -> 3D opening -> bridge-kill ->
           dust removal -> 2D hole fill -> XY opening -> fallback guards."""
    tag = "    [PP]" if verbose else ""

    # 1. Smoothing
    prob = gaussian_filter(prob, sigma=PP_SMOOTH_SIGMA).astype(np.float32)
    prob = np.clip(prob, 0, 1)

    # 2. Hysteresis thresholding
    mask = hysteresis_3d(prob, tl, th)
    if verbose:
        print(f"{tag} After hysteresis(tl={tl:.2f},th={th:.2f}): {mask.sum()} FG voxels")

    # 3. Fallback guard: empty mask -> lower thresholds
    n_retries = 0
    retry_tl, retry_th = tl, th
    while mask.sum() == 0 and n_retries < 3:
        retry_tl = max(0.15, retry_tl - PP_EMPTY_TL_DROP)
        retry_th = max(0.30, retry_th - PP_EMPTY_TH_DROP)
        mask = hysteresis_3d(prob, retry_tl, retry_th)
        n_retries += 1
        if verbose:
            print(f"{tag} Empty mask! Retry {n_retries}: tl={retry_tl:.2f}, th={retry_th:.2f} -> {mask.sum()} FG")

    # 4. 3D morphological opening (removes thin 3D bridges/spurs)
    if PP_OPEN_R_3D > 0:
        before = mask.sum()
        mask = opening_3d(mask, PP_OPEN_R_3D)
        if verbose and mask.sum() != before:
            print(f"{tag} 3D opening(r={PP_OPEN_R_3D}): {before} -> {mask.sum()} FG")

    # 5. Bridge killer
    if PP_BRIDGE_KILL:
        before = mask.sum()
        mask = bridge_killer(mask)
        if verbose and mask.sum() != before:
            print(f"{tag} Bridge-kill: {before} -> {mask.sum()} FG")

    # 6. Dust removal (after bridge-kill to catch fragments)
    before = mask.sum()
    mask = remove_dust_3d(mask)
    if verbose and mask.sum() != before:
        print(f"{tag} Dust removal(min={PP_DUST_MIN_3D}): {before} -> {mask.sum()} FG")

    # 7. 2D hole filling
    mask = fill_holes_2d(mask)

    # 8. XY opening
    if PP_OPEN_R_XY > 0:
        mask = xy_opening(mask)

    # 9. Fallback guard: overfull mask -> raise threshold
    fg_frac = mask.sum() / max(mask.size, 1)
    if fg_frac > PP_OVERFULL_FRAC:
        raised_th = th + PP_OVERFULL_TH_RAISE
        if verbose:
            print(f"{tag} Overfull ({fg_frac:.3f})! Re-thresholding with th={raised_th:.2f}")
        mask = hysteresis_3d(prob, tl, raised_th)
        mask = remove_dust_3d(mask)
        mask = fill_holes_2d(mask)

    return mask


def postprocess_single_model(prob, tl, th, dust_min=None):
    """Per-model postprocess for vote ensemble: binarize + full cleanup.
    Used when ENS_MODE='vote' to clean each model's mask independently.
    v2.4: per-model dust_min override from meta."""
    old_dust = PP_DUST_MIN_3D
    if dust_min is not None:
        # Temporarily override global dust for this model
        globals()["PP_DUST_MIN_3D"] = dust_min
    result = postprocess(prob, tl, th, verbose=False)
    globals()["PP_DUST_MIN_3D"] = old_dust
    return result


# v2.4b: Weighted median thresholds (resistant to outlier models pulling thresholds)
def _weighted_median(values, weights):
    """Compute weighted median: resists outlier models dragging thresholds."""
    if not values:
        return 0.5
    pairs = sorted(zip(values, weights))
    cumw = np.cumsum([w for _, w in pairs])
    total = cumw[-1]
    idx = np.searchsorted(cumw, total * 0.5)
    idx = min(idx, len(pairs) - 1)
    return pairs[idx][0]

_tls, _ths, _weights = [], [], []
for mn, meta in metas.items():
    if mn not in models:
        continue
    _tls.append(meta.get("tl", DEFAULT_TL))
    _ths.append(meta.get("th", DEFAULT_TH))
    # v2.4b: Use combined_proxy as weight (primary trust metric)
    proxy = meta.get("combined_proxy", meta.get("best_val_dice", 0.5))
    _weights.append(max(float(proxy), 0.1))

if _weights:
    ENS_TL = _weighted_median(_tls, _weights)
    ENS_TH = _weighted_median(_ths, _weights)
else:
    ENS_TL, ENS_TH = DEFAULT_TL, DEFAULT_TH

# Per-model thresholds for vote mode
PER_MODEL_TL = {}
PER_MODEL_TH = {}
for mn, meta in metas.items():
    if mn not in models:
        continue
    PER_MODEL_TL[mn] = meta.get("tl", DEFAULT_TL)
    PER_MODEL_TH[mn] = meta.get("th", DEFAULT_TH)

print(f"[PP] Ensemble thresholds: tl={ENS_TL:.3f}, th={ENS_TH:.3f}")
print(f"[PP] Per-model TL: {PER_MODEL_TL}")
print(f"[PP] Per-model TH: {PER_MODEL_TH}")
print(f"[PP] Model weights: {dict(zip([m for m in metas if m in models], [f'{w:.3f}' for w in _weights]))}")


In [ ]:

# ============================================================
# Ensemble Sliding-Window Inference
# v2.2: per-model binarize+cleanup -> majority vote (domination mode)
#        + confidence sharpening, logit fusion fallback, TTA
# ============================================================

def normalize_volume(vol):
    v = vol.astype(np.float32)
    mu = v.mean()
    sd = v.std() + 1e-8
    return (v - mu) / sd


def _gauss_1d(n):
    if n <= 1: return np.ones(n, dtype=np.float32)
    x = np.linspace(-1, 1, n, dtype=np.float32)
    return np.exp(-2.0 * x * x)


def _gauss_3d(shape):
    w = (_gauss_1d(shape[0])[:, None, None] *
         _gauss_1d(shape[1])[None, :, None] *
         _gauss_1d(shape[2])[None, None, :])
    return w / (w.max() + 1e-8)


def sharpen_probs(prob, temperature):
    """Confidence sharpening: temperature < 1.0 makes predictions more decisive.
    Sharpens before binarization to reduce boundary ambiguity."""
    if temperature == 1.0 or temperature <= 0:
        return prob
    p_clip = np.clip(prob, 1e-6, 1 - 1e-6)
    logit = np.log(p_clip / (1 - p_clip)) / temperature
    return 1.0 / (1.0 + np.exp(-logit))


@torch.no_grad()
def infer_single_model(vol_f32, model_m, roi, overlap, use_tta=False):
    """Sliding window inference. Returns FG probability (D,H,W)."""
    D, H, W = vol_f32.shape
    rD, rH, rW = roi
    sD = max(1, int(rD * (1.0 - overlap)))
    sH = max(1, int(rH * (1.0 - overlap)))
    sW = max(1, int(rW * (1.0 - overlap)))
    w3d = _gauss_3d((rD, rH, rW))

    acc = np.zeros((D, H, W), dtype=np.float32)
    wacc = np.zeros((D, H, W), dtype=np.float32)

    z_starts = sorted(set(list(range(0, max(1, D-rD+1), sD)) + [max(0, D-rD)]))
    y_starts = sorted(set(list(range(0, max(1, H-rH+1), sH)) + [max(0, H-rH)]))
    x_starts = sorted(set(list(range(0, max(1, W-rW+1), sW)) + [max(0, W-rW)]))

    amp_on = (DEVICE.type == "cuda")
    is_half = next(model_m.parameters()).dtype == torch.float16

    def _run_patch(patch_np):
        t = torch.from_numpy(patch_np[None, None].copy())
        if is_half:
            t = t.half()
        t = t.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp_on):
            logits = model_m(t)
            if isinstance(logits, (list, tuple)):
                logits = logits[0]
        probs = torch.softmax(logits[0].float(), dim=0)
        return probs[1].cpu().numpy()

    for z0 in z_starts:
        for y0 in y_starts:
            for x0 in x_starts:
                patch = vol_f32[z0:z0+rD, y0:y0+rH, x0:x0+rW]
                if patch.shape != (rD, rH, rW):
                    continue

                fg_prob = _run_patch(patch)

                if use_tta:
                    n_tta = 1
                    for ax in [2, 1]:
                        flipped = np.flip(patch, axis=ax).copy()
                        fp = _run_patch(flipped)
                        fg_prob = fg_prob + np.flip(fp, axis=ax)
                        n_tta += 1

                    for k in [1, 2, 3]:
                        rotated = np.rot90(patch, k, axes=(1, 2)).copy()
                        if rotated.shape == (rD, rH, rW):
                            fp = _run_patch(rotated)
                            fg_prob = fg_prob + np.rot90(fp, -k, axes=(1, 2))
                            n_tta += 1

                    fg_prob /= n_tta

                acc[z0:z0+rD, y0:y0+rH, x0:x0+rW] += fg_prob * w3d
                wacc[z0:z0+rD, y0:y0+rH, x0:x0+rW] += w3d

    wacc = np.maximum(wacc, 1e-8)
    return acc / wacc


def vote_ensemble(per_model_probs, use_tta):
    """
    Per-model binarize + cleanup -> majority vote.
    Each model's probs get sharpened, thresholded with its own calibrated
    thresholds, and fully post-processed independently. Then a per-voxel
    majority vote decides the final mask.
    This avoids bridge-fattening from logit averaging: if one model has a
    spurious bridge, the other two vote it out.
    Quorum: ceil(n/2) = 2/3 for 3 models, 1/1 for 1 model, 2/2 for 2 models.
    """
    names = list(per_model_probs.keys())
    n = len(names)
    quorum = max(1, (n + 1) // 2)  # ceil(n/2): 2/3, 1/1, 2/2

    if n == 1:
        # Single model: just postprocess normally
        nm = names[0]
        tl = PER_MODEL_TL.get(nm, DEFAULT_TL)
        th = PER_MODEL_TH.get(nm, DEFAULT_TH)
        prob = sharpen_probs(per_model_probs[nm], CONF_TEMPERATURE)
        return postprocess(prob, tl, th, verbose=True)

    # Per-model binarize + cleanup (v2.4: per-model dust from meta)
    per_model_masks = {}
    for nm in names:
        tl = PER_MODEL_TL.get(nm, DEFAULT_TL)
        th = PER_MODEL_TH.get(nm, DEFAULT_TH)
        dust = metas.get(nm, {}).get("dust_min", PP_DUST_MIN_3D)
        prob = sharpen_probs(per_model_probs[nm], CONF_TEMPERATURE)
        mask = postprocess_single_model(prob, tl, th, dust_min=dust)
        per_model_masks[nm] = mask
        print(f"    {nm}: post-proc FG={mask.sum()} voxels "
              f"(tl={tl:.2f}, th={th:.2f}, dust={dust})")

    # Majority vote
    vote_sum = np.zeros_like(next(iter(per_model_masks.values())), dtype=np.int32)
    for mask in per_model_masks.values():
        vote_sum += mask.astype(np.int32)

    voted = (vote_sum >= quorum).astype(np.uint8)
    print(f"    Vote: quorum={quorum}/{n}, FG={voted.sum()} voxels")

    # Light cleanup on voted result (dust only, no aggressive postproc)
    voted = remove_dust_3d(voted, min_size=PP_DUST_MIN_3D)

    return voted


def outlier_suppress_weights(per_model_probs, model_weights):
    """v2.4: Per-volume outlier suppression with TWO signals:
    1. FG fraction: if one model's FG fraction is wildly different from median
    2. Component count: if one model produces far more components (fragmentation)
    Prevents one crazy model from poisoning the ensemble on a specific volume."""
    names = list(per_model_probs.keys())
    if len(names) < 3:
        return dict(model_weights)  # Need 3+ models for meaningful outlier detection

    struct26 = generate_binary_structure(3, 3)
    adjusted = dict(model_weights)

    # Signal 1: FG fraction outlier
    fg_fracs = {nm: float((per_model_probs[nm] > 0.5).mean()) for nm in names}
    median_fg = float(np.median(list(fg_fracs.values())))

    for nm in names:
        # v2.4b: Near-empty mask downweight (model failed on this volume)
        if fg_fracs[nm] < 1e-5 and median_fg > 1e-4:
            adjusted[nm] = model_weights[nm] * 0.15
            print(f"    [OUTLIER-EMPTY] {nm}: near-empty mask, "
                  f"weight {model_weights[nm]:.3f} -> {adjusted[nm]:.3f}")
            continue
        if median_fg < 1e-6:
            continue
        ratio = fg_fracs[nm] / max(median_fg, 1e-6)
        if ratio > 3.0 or ratio < 0.33:
            adjusted[nm] = model_weights[nm] * 0.3
            print(f"    [OUTLIER-FG] {nm}: fg_frac={fg_fracs[nm]:.4f} vs median={median_fg:.4f}, "
                  f"weight {model_weights[nm]:.3f} -> {adjusted[nm]:.3f}")
        elif ratio > 2.0 or ratio < 0.5:
            adjusted[nm] = model_weights[nm] * 0.6

    # Signal 2: Component count outlier (fragmentation detector)
    # Quick component count on thresholded prob at 0.5
    comp_counts = {}
    for nm in names:
        binary = (per_model_probs[nm] > 0.5).astype(np.uint8)
        if binary.sum() > 0:
            _, n_cc = cc_label(binary, structure=struct26)
            comp_counts[nm] = n_cc
        else:
            comp_counts[nm] = 0

    if comp_counts:
        median_cc = float(np.median(list(comp_counts.values())))
        for nm in names:
            if median_cc < 1:
                continue
            cc_ratio = comp_counts[nm] / max(median_cc, 1)
            # If model has >5x median components, it's fragmenting badly
            if cc_ratio > 5.0:
                penalty = 0.25
                new_w = min(adjusted[nm], model_weights[nm] * penalty)
                if new_w < adjusted[nm]:
                    print(f"    [OUTLIER-CC] {nm}: {comp_counts[nm]} components vs median={median_cc:.0f}, "
                          f"weight -> {new_w:.3f}")
                    adjusted[nm] = new_w
            elif cc_ratio > 3.0:
                penalty = 0.5
                adjusted[nm] = min(adjusted[nm], model_weights[nm] * penalty)

    return adjusted


def logit_fuse_ensemble(per_model_probs, model_weights):
    """Primary: logit-space weighted average -> probability -> postprocess."""
    names = list(per_model_probs.keys())
    n = len(names)

    if n == 1:
        prob = sharpen_probs(next(iter(per_model_probs.values())), CONF_TEMPERATURE)
        return postprocess(prob, ENS_TL, ENS_TH, verbose=True)

    # v2.4: Per-volume outlier suppression before fusion
    vol_weights = outlier_suppress_weights(per_model_probs, model_weights)

    w_total = sum(vol_weights[nm] for nm in names)
    result = np.zeros_like(next(iter(per_model_probs.values())))
    for nm in names:
        # v2.4: Per-model temperature already applied during inference (ensemble_predict).
        # Don't double-sharpen here. Just convert to logits for fusion.
        p = np.clip(per_model_probs[nm], 1e-6, 1 - 1e-6)
        logit = np.log(p / (1 - p))
        result += logit * (vol_weights[nm] / w_total)

    # v2.4: Apply global confidence sharpening ONCE on fused logits
    if CONF_TEMPERATURE != 1.0 and CONF_TEMPERATURE > 0:
        result = result / CONF_TEMPERATURE

    fused_prob = 1.0 / (1.0 + np.exp(-np.clip(result, -50, 50)))
    return postprocess(fused_prob, ENS_TL, ENS_TH, verbose=True)


def compute_auto_weights(model_names, metas_dict):
    """v2.4: Auto-weights from proxy scores (softmax over normalized proxy).
    Uses combined proxy if available, else val_dice, with role multiplier as tiebreaker.
    Clamps so no model becomes 0 (min 0.1)."""
    raw = {}
    for mn in model_names:
        meta = metas_dict.get(mn, {})
        # Prefer combined proxy (calibration-derived), fallback to val_dice
        proxy = meta.get("combined_proxy", meta.get("best_val_dice", 0.3))
        role = meta.get("model_role", "base")
        role_mult = ROLE_WEIGHTS.get(role, 1.0)
        raw[mn] = max(float(proxy) * role_mult, 0.05)

    # Softmax normalization for smooth weight distribution
    vals = np.array([raw[mn] for mn in model_names])
    # Temperature=1.0 for softmax (higher spreads weights more evenly)
    exp_vals = np.exp(vals - vals.max())  # Numerical stability
    weights = exp_vals / (exp_vals.sum() + 1e-8)
    # Scale to sum=len so average weight=1, clamp min 0.1
    weights = weights * len(model_names)
    weights = np.clip(weights, 0.1, None)

    return {mn: float(w) for mn, w in zip(model_names, weights)}


def ensemble_predict(vol_u8, use_tta):
    """Run all models with per-model ROI, then fuse via configured mode."""
    vol_f32 = normalize_volume(vol_u8)

    # v2.4: Auto-weights from proxy scores (data-driven, not hardcoded)
    model_weights = compute_auto_weights(list(models.keys()), metas)

    per_model_probs = {}

    # v2.4: Selective TTA — only the weakest model(s) get TTA if enabled.
    # This halves runtime vs full TTA while stabilizing the least reliable model.
    if use_tta and len(models) >= 2:
        # v2.4b: Sort by combined_proxy (ascending), TTA for bottom half
        proxies = {mn: metas.get(mn, {}).get("combined_proxy",
                       metas.get(mn, {}).get("best_val_dice", 0.3)) for mn in models}
        sorted_models = sorted(proxies.keys(), key=lambda x: proxies[x])
        n_tta = max(1, len(sorted_models) // 2)  # TTA for weakest half
        tta_set = set(sorted_models[:n_tta])
    else:
        tta_set = set(models.keys()) if use_tta else set()

    for mname, m in models.items():
        # v2.4: Per-model ROI and overlap from meta (match training distribution)
        roi = tuple(metas.get(mname, {}).get("patch_size", list(DEFAULT_ROI)))
        roi_clamped = tuple(min(r, s) for r, s in zip(roi, vol_f32.shape))
        model_overlap = metas.get(mname, {}).get("inf_overlap", INF_OVERLAP)

        model_tta = mname in tta_set
        t0 = time.time()
        prob = infer_single_model(vol_f32, m, roi_clamped, model_overlap, model_tta)
        dt = time.time() - t0

        # Per-model temperature from calibration
        temp_cal = metas.get(mname, {}).get("temperature", 1.0)
        if temp_cal != 1.0 and temp_cal > 0:
            prob = sharpen_probs(prob, temp_cal)

        per_model_probs[mname] = prob
        w = model_weights[mname]
        tta_tag = " +TTA" if model_tta else ""
        print(f"    {mname}: {dt:.1f}s, w={w:.3f}, roi={roi_clamped}, "
              f"prob=[{prob.min():.3f}, {prob.max():.3f}]{tta_tag}")

    # Fuse using configured mode
    if ENS_MODE == "vote" and len(per_model_probs) >= 3:
        return vote_ensemble(per_model_probs, use_tta)
    else:
        # Primary: logit fusion → single global postproc (recommended by researcher)
        return logit_fuse_ensemble(per_model_probs, model_weights)


In [ ]:

# ============================================================
# Main Inference Loop
# v2.2: run manifest, sanity checks, always uint8 {0,1}
# ============================================================
OUT_DIR = "/kaggle/working/preds"
os.makedirs(OUT_DIR, exist_ok=True)
ZIP_PATH = "/kaggle/working/submission.zip"

# v2.2: Run manifest
submission_meta = {
    "version": "v2.5",
    "models": list(models.keys()),
    "model_paths": {k: MODEL_PATHS[k] for k in models},
    "ens_mode": ENS_MODE,
    "ens_tl": ENS_TL,
    "ens_th": ENS_TH,
    "conf_temperature": CONF_TEMPERATURE,
    "default_roi": list(DEFAULT_ROI),
    "overlap": INF_OVERLAP,
    "pp_dust_min_3d": PP_DUST_MIN_3D,
    "pp_hole_max_2d": PP_HOLE_MAX_2D,
    "pp_open_r_xy": PP_OPEN_R_XY,
    "pp_open_r_3d": PP_OPEN_R_3D,
    "pp_smooth_sigma": PP_SMOOTH_SIGMA,
    "pp_bridge_kill": PP_BRIDGE_KILL,
    "pp_bk_neck_r": PP_BK_NECK_R,
    "pp_bk_min_lobe": PP_BK_MIN_LOBE,
    "auto_drop_thresh": AUTO_DROP_THRESH,
    "pytorch_version": torch.__version__,
    "metas": {k: {kk: vv for kk, vv in v.items()
                   if isinstance(vv, (int, float, str, bool))}
              for k, v in metas.items() if k in models},
}

n_test = len(test_ids)
time_per_vol_budget = (MAX_PRED_HOURS * 3600 - 300) / max(n_test, 1)
print(f"[INF] {n_test} volumes, budget {time_per_vol_budget:.0f}s each")

# Adaptive TTA
current_tta = USE_TTA
n_models = len(models)
est_per_vol = n_models * 120 * (6 if current_tta else 1)
if est_per_vol > time_per_vol_budget * 0.8:
    current_tta = False
    print(f"[INF] TTA disabled (est {est_per_vol:.0f}s > budget {time_per_vol_budget:.0f}s)")

completed = 0
per_vol_stats = []

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for vi, (vid, vpath) in enumerate(zip(test_ids, test_paths)):
        if not budget_ok(MAX_PRED_HOURS - 0.1):
            print(f"\n[TIME] Budget near limit. {completed}/{n_test} done.")
            break

        t0 = time.time()
        print(f"\n[{vi+1}/{n_test}] {vid}...")

        try:
            vol = read_tif(vpath)
            orig_shape = vol.shape
            print(f"  Shape: {orig_shape}, dtype: {vol.dtype}")

            # ensemble_predict returns final mask (vote mode) or postprocessed mask
            mask = ensemble_predict(vol, current_tta)

            # Always uint8 {0,1}
            assert mask.shape == orig_shape, f"Shape mismatch: {mask.shape} vs {orig_shape}"
            mask = mask.astype(np.uint8)
            mask = np.clip(mask, 0, 1)

            # Sanity checks
            fg_frac = mask.sum() / mask.size
            if fg_frac == 0:
                print(f"  [WARN] Empty prediction for {vid}!")
            elif fg_frac > 0.5:
                print(f"  [WARN] Very high FG fraction: {fg_frac:.4f}")

            out_path = os.path.join(OUT_DIR, f"{vid}.tif")
            write_tif(out_path, mask)
            zf.write(out_path, f"{vid}.tif")

            dt = time.time() - t0
            completed += 1
            per_vol_stats.append({"id": vid, "time_s": dt, "fg_frac": float(fg_frac),
                                  "shape": list(orig_shape)})
            print(f"  Done: {dt:.1f}s, FG={fg_frac:.4f}")

            # v2.5: Full degradation ladder (TTA -> overlap -> always finish)
            remaining = n_test - vi - 1
            if remaining > 0:
                remain_t = (MAX_PRED_HOURS - elapsed_h()) * 3600
                est_remaining = remaining * dt
                if current_tta and est_remaining > remain_t * 0.9:
                    current_tta = False
                    print("  [ADAPT] TTA disabled to stay in budget")
                elif not current_tta and est_remaining > remain_t * 0.85:
                    INF_OVERLAP = max(0.25, INF_OVERLAP * 0.75)
                    print(f"  [ADAPT] Overlap reduced to {INF_OVERLAP:.2f}")

        except Exception as e:
            print(f"  [ERROR] {vid}: {e}")
            traceback.print_exc()
            try:
                empty = np.zeros(orig_shape, dtype=np.uint8)
                out_path = os.path.join(OUT_DIR, f"{vid}.tif")
                write_tif(out_path, empty)
                zf.write(out_path, f"{vid}.tif")
                completed += 1
                per_vol_stats.append({"id": vid, "time_s": 0, "fg_frac": 0.0, "error": str(e)})
            except Exception:
                pass

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

# Save submission metadata
submission_meta["completed"] = completed
submission_meta["total_time_h"] = elapsed_h()
submission_meta["per_volume"] = per_vol_stats
meta_out = "/kaggle/working/submission_meta.json"
with open(meta_out, "w") as f:
    json.dump(submission_meta, f, indent=2)
print(f"\n[META] Saved: {meta_out}")

print(f"\n[DONE] {completed}/{n_test} volumes in {elapsed_h():.2f}h")
print(f"  Submission: {ZIP_PATH}")


In [ ]:

# ============================================================
# Submission Validation
# ============================================================

assert os.path.exists(ZIP_PATH), f"submission.zip not found at {ZIP_PATH}"
assert os.path.basename(ZIP_PATH) == "submission.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    assert len(names) > 0, "submission.zip is empty"
    print(f"[ZIP] {len(names)} files in submission.zip")
    for n in names[:5]:
        info = zf.getinfo(n)
        print(f"  {n}: {info.file_size/1e6:.1f} MB")
    if len(names) > 5:
        print(f"  ... and {len(names)-5} more")

    # v2.2: Verify all files are valid uint8 TIFs
    for n in names[:3]:
        with zf.open(n) as fp:
            header = fp.read(4)
            # TIFF magic bytes: II (little-endian) or MM (big-endian)
            assert header[:2] in (b'II', b'MM'), f"{n}: Not a valid TIFF file (header={header[:2]})"
    print(f"  TIFF headers verified")

    # v2.5: Verify all test volumes have predictions
    zip_basenames = set(os.path.basename(n).replace(".tif", "") for n in names)
    missing = [tid for tid in test_ids if tid not in zip_basenames]
    if missing:
        print(f"  [WARN] Missing predictions for {len(missing)} volumes: {missing[:5]}")
    else:
        print(f"  All {len(test_ids)} test volumes have predictions")

sz = os.path.getsize(ZIP_PATH) / 1e6
print(f"\n[OK] submission.zip: {sz:.1f} MB, {len(names)} volumes")
print(f"[OK] Total time: {elapsed_h():.2f}h")

# Print submission metadata
if os.path.exists("/kaggle/working/submission_meta.json"):
    with open("/kaggle/working/submission_meta.json") as f:
        sm = json.load(f)
    print(f"\n[META] Version: {sm.get('version')}")
    print(f"[META] Models: {sm.get('models')}")
    print(f"[META] Fusion: {sm.get('ens_mode')}")
    print(f"[META] Thresholds: tl={sm.get('ens_tl'):.3f}, th={sm.get('ens_th'):.3f}")

    # FG fraction summary
    stats = sm.get("per_volume", [])
    if stats:
        fg_fracs = [s["fg_frac"] for s in stats if "fg_frac" in s]
        if fg_fracs:
            print(f"[META] FG fraction: min={min(fg_fracs):.4f}, max={max(fg_fracs):.4f}, "
                  f"mean={np.mean(fg_fracs):.4f}")
        errors = [s for s in stats if "error" in s]
        if errors:
            print(f"[META] {len(errors)} volume(s) had errors")

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
